In [8]:
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# -----------------------------
# 1. Load assessment datasets
# -----------------------------
embic = pd.read_csv("Assessments/ADNI_EMBICDCB_08Jul2025.csv")
moca = pd.read_csv("Assessments/MOCA_08Jul2025.csv")
dxsum = pd.read_csv("Assessments/DXSUM_08Jul2025.csv")  # ground truth labels

# -----------------------------
# 2. Merge assessments (EMBIC + MOCA)
# -----------------------------
data = pd.merge(embic, moca, on=["RID", "VISCODE2"], how="inner")

# -----------------------------
# 3. Add labels from DXSUM
# -----------------------------
# Keep only columns we need from DXSUM
dxsum_labels = dxsum[["RID", "VISCODE2", "DXCURREN"]]

# Merge into dataset
data = pd.merge(data, dxsum_labels, on=["RID", "VISCODE2"], how="inner")

# -----------------------------
# 4. Clean data
# -----------------------------
meta_cols = [
    "RID", "VISCODE2", "update_stamp_x", "update_stamp_y",
    "PHASE", "PTID", "VISCODE", "VISDATE", "EXAMDATE",
    "USERDATE", "USERDATE2", "DD_CRF_VERSION_LABEL",
    "LANGUAGE_CODE", "HAS_QC_ERROR", "SOURCE", "ID", "SITEID"
]

# Drop metadata if present
X = data.drop(columns=[c for c in meta_cols if c in data.columns])

# Separate target
y = X["DXCURREN"]   # CN / MCI / AD
X = X.drop(columns=["DXCURREN"])

# Keep only numeric features
X = X.select_dtypes(include=["int64", "float64"])

# Drop columns that are fully empty
X = X.dropna(axis=1, how="all")

# Drop rows with missing labels
valid_idx = y.notna()
X = X.loc[valid_idx]
y = y.loc[valid_idx]

# -----------------------------
# 5. Encode target variable
# -----------------------------
# CN=0, MCI=1, AD=2
label_map = {"CN": 0, "MCI": 1, "AD": 2}
y = y.map(label_map)

# -----------------------------
# 6. Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# 7. Random Forest Model
# -----------------------------
model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train, y_train)

# -----------------------------
# 8. Performance
# -----------------------------
print("Classification Report:\n")
print(classification_report(y_test, model.predict(X_test), target_names=label_map.keys()))

# -----------------------------
# 9. SHAP Explainability
# -----------------------------
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Plot top features
shap.summary_plot(shap_values, X_test, plot_type="bar")


KeyError: "['DXCURREN'] not in index"

In [9]:
print(dxsum.columns.tolist())


['PHASE', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE', 'DIAGNOSIS', 'DXNORM', 'DXNODEP', 'DXMCI', 'DXMDES', 'DXMPTR1', 'DXMPTR2', 'DXMPTR3', 'DXMPTR4', 'DXMPTR5', 'DXMPTR6', 'DXMDUE', 'DXMOTHET', 'DXDSEV', 'DXDDUE', 'DXAD', 'DXAPP', 'DXAPROB', 'DXAPOSS', 'DXPARK', 'DXPDES', 'DXPCOG', 'DXPATYP', 'DXDEP', 'DXOTHDEM', 'DXODES', 'DXCONFID', 'ID', 'SITEID', 'USERDATE', 'USERDATE2', 'DD_CRF_VERSION_LABEL', 'LANGUAGE_CODE', 'HAS_QC_ERROR', 'update_stamp']


In [11]:
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# -----------------------------
# 1. Load data
# -----------------------------
embic = pd.read_csv("Assessments/EMBIC.csv")   # adjust filename if different
moca = pd.read_csv("Assessments/MOCA.csv")     # adjust filename if different
dxsum = pd.read_csv("Assessments/DXSUM.csv")   # contains diagnosis labels

# -----------------------------
# 2. Merge EMBIC + MOCA
# -----------------------------
data = pd.merge(embic, moca, on=["RID", "VISCODE2"], how="inner")

# -----------------------------
# 3. Add labels from DXSUM
# -----------------------------
dxsum_labels = dxsum[["RID", "VISCODE2", "DIAGNOSIS"]]

# Merge into dataset
data = pd.merge(data, dxsum_labels, on=["RID", "VISCODE2"], how="inner")

# -----------------------------
# 4. Prepare Features (X) and Labels (y)
# -----------------------------
# Drop metadata columns if they exist
drop_cols = [c for c in ["RID", "VISCODE2", "PHASE", "PTID", "VISCODE", "VISDATE", "update_stamp"] if c in data.columns]
X = data.drop(columns=drop_cols + ["DIAGNOSIS"])

# Labels
# Typically DIAGNOSIS has values like CN/MCI/AD. Let's map:
label_map = {
    "CN": 0,    # Cognitively Normal
    "MCI": 1,   # Mild Cognitive Impairment
    "AD": 2     # Alzheimer's Disease
}
y = data["DIAGNOSIS"].map(label_map)

# -----------------------------
# 5. Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# 6. Train Random Forest
# -----------------------------
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# -----------------------------
# 7. Evaluate
# -----------------------------
print(classification_report(y_test, model.predict(X_test)))

# -----------------------------
# 8. Explain with SHAP
# -----------------------------
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Bar summary plot of most important features
shap.summary_plot(shap_values, X_test, plot_type="bar")

# Optional: detailed beeswarm plot
shap.summary_plot(shap_values, X_test)


FileNotFoundError: [Errno 2] No such file or directory: 'Assessments/EMBIC.csv'

In [ ]:
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()

embic_path = filedialog.askopenfilename(title="Select EMBIC.csv")
moca_path = filedialog.askopenfilename(title="Select MOCA.csv")
dxsum_path = filedialog.askopenfilename(title="Select DXSUM.csv")

embic = pd.read_csv(embic_path)
moca = pd.read_csv(moca_path)
dxsum = pd.read_csv(dxsum_path)


In [4]:
# =========================
# Install (only first time)
# =========================
# pip install pandas numpy scikit-learn shap matplotlib pypdf

import re
import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pypdf import PdfReader
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import shap

# -------------------------
# 0) CONFIG: edit paths
# -------------------------
PDF_PATH = "docs/ADAS_EMBIC_qCP_Methods.pdf"      # <- your methods PDF
DATA_DIR = "Assessments"                           # <- folder that holds CSVs
# If your filenames are different, you can force them here; else we auto-detect:
EMBIC_GLOB = ["*EMBIC*.csv", "*EMBIC*.xlsx", "ADNI_EMBICDCB_*.csv"]
MOCA_GLOB  = ["*MOCA*.csv", "*MOCA*.xlsx", "MOCA_*.csv"]
DX_GLOB    = ["*DXSUM*.csv", "*DXSUM*.xlsx", "DX*.csv"]

# -------------------------
# 1) Parse PDF to infer features
# -------------------------
def parse_pdf_for_feature_hints(pdf_path: str) -> dict:
    """
    Read the methods PDF and infer which families of features to look for in CSVs.
    We search for terms like 'N1 N2 N3 N4', 'R1 R2 R3', 'M1 M2 M3', and ADAS/MoCA keywords.
    """
    hints = {
        "qcp_encoding": [],   # N1-N4
        "qcp_retrieval": [],  # R1-R3
        "qcp_behavioral": [], # M1-M3 or M* names
        "moca_like": [],      # MoCA subtests (Trails, Cube, Clock*, Digits, Serial*, etc.)
        "adas_like": []       # optional ADAS subscores if present
    }
    try:
        reader = PdfReader(pdf_path)
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
    except Exception as e:
        print(f"[WARN] Could not read PDF ({pdf_path}): {e}")
        text = ""

    # Look for explicit tokens
    # Heuristics: capture N1-4, R1-3, M1-3 (or M-prefixed items)
    n_matches = sorted(set(re.findall(r"\bN([1-4])\b", text)))
    r_matches = sorted(set(re.findall(r"\bR([1-3])\b", text)))
    m_matches = sorted(set(re.findall(r"\bM([1-9])\b", text)))  # some docs use M1..M3, but allow >3

    hints["qcp_encoding"] = [f"N{i}" for i in range(1, 5)] if n_matches else ["N1","N2","N3","N4"]
    hints["qcp_retrieval"] = [f"R{i}" for i in range(1, 4)] if r_matches else ["R1","R2","R3"]
    # We don't know exact M count; assume up to M3 by default
    if m_matches:
        max_m = max(int(i) for i in m_matches)
        hints["qcp_behavioral"] = [f"M{i}" for i in range(1, max_m+1)]
    else:
        hints["qcp_behavioral"] = ["M1","M2","M3"]

    # MoCA-like tokens
    moca_tokens = [
        "TRAILS","CUBE","CLOCKCON","CLOCKNO","CLOCKHAN",
        "LION","RHINO","CAMEL",
        "IMMT1W1","IMMT1W2","IMMT1W3","IMMT1W4","IMMT1W5",
        "IMMT2W1","IMMT2W2","IMMT2W3","IMMT2W4","IMMT2W5",
        "DIGFOR","DIGBACK","LETTERS",
        "SERIAL1","SERIAL2","SERIAL3","SERIAL4","SERIAL5",
        "REPEAT1","REPEAT2","FFLUENCY","ABSTRAN","ABSMEAS",
        "DELW1","DELW2","DELW3","DELW4","DELW5",
        "DATE","MONTH","YEAR","DAY","PLACE","CITY","MOCA"
    ]
    hints["moca_like"] = moca_tokens

    # ADAS-like (optional; often embedded elsewhere)
    hints["adas_like"] = ["ADAS", "WORD", "RECALL", "RECOGNITION"]

    return hints

feature_hints = parse_pdf_for_feature_hints(PDF_PATH)
print("[INFO] Feature hints inferred from PDF:")
print(json.dumps(feature_hints, indent=2))

# -------------------------
# 2) Find and load files
# -------------------------
def first_match(globs, base_dir):
    for pat in globs:
        hits = glob.glob(os.path.join(base_dir, pat))
        if hits:
            return hits[0]
    return None

def load_tabular(path):
    if path is None:
        return None
    ext = os.path.splitext(path)[1].lower()
    if ext in [".csv", ".txt"]:
        return pd.read_csv(path)
    elif ext in [".xlsx", ".xls"]:
        try:
            return pd.read_excel(path)
        except Exception:
            # fallback for odd encodings
            return pd.read_csv(path)
    else:
        return pd.read_csv(path)

embic_path = first_match(EMBIC_GLOB, DATA_DIR)
moca_path  = first_match(MOCA_GLOB, DATA_DIR)
dx_path    = first_match(DX_GLOB, DATA_DIR)

print(f"[INFO] EMBIC file: {embic_path}")
print(f"[INFO] MOCA  file: {moca_path}")
print(f"[INFO] DXSUM file: {dx_path}")

if not embic_path or not moca_path or not dx_path:
    raise FileNotFoundError("Could not auto-detect one or more files. Update CONFIG paths at the top.")

embic = load_tabular(embic_path)
moca  = load_tabular(moca_path)
dxsum = load_tabular(dx_path)

# -------------------------
# 3) Minimal schema checks
# -------------------------
key_candidates = [("RID","VISCODE2"), ("RID","VISCODE"), ("RID","VISCODE2_x")]
def find_keys(df, keys=key_candidates):
    for k1, k2 in keys:
        if k1 in df.columns and k2 in df.columns:
            return [k1, k2]
    # fallback: just RID if single timepoint
    if "RID" in df.columns:
        return ["RID"]
    raise KeyError("No suitable keys found. Expected columns like RID and VISCODE2.")

keys_embic = find_keys(embic)
keys_moca  = find_keys(moca)
# For DXSUM we expect RID + VISCODE2, else try to degrade gracefully
keys_dx    = find_keys(dxsum)

# -------------------------
# 4) Merge datasets
# -------------------------
merged = pd.merge(embic, moca, left_on=keys_embic, right_on=keys_moca, how="inner", suffixes=("_embic","_moca"))

# Pick a likely label column from DXSUM: prefer DIAGNOSIS else look for DX*, else fail
label_col = None
preferred_labels = ["DIAGNOSIS", "DXCURREN", "DXCHANGE", "DX", "DXSTATUS"]
for cand in preferred_labels:
    if cand in dxsum.columns:
        label_col = cand
        break
if label_col is None:
    # heuristic: first column starting with 'DX' that looks categorical
    dx_like = [c for c in dxsum.columns if c.upper().startswith("DX")]
    if dx_like:
        label_col = dx_like[0]
if label_col is None:
    raise KeyError("Could not find a diagnosis label column in DXSUM (e.g., DIAGNOSIS/DXCURREN/DXCHANGE).")

dx_keep = keys_dx + [label_col]
dx_small = dxsum[dx_keep].copy()

merged = pd.merge(merged, dx_small, left_on=keys_embic, right_on=keys_dx, how="inner")

# -------------------------
# 5) Build candidate features based on PDF hints
# -------------------------
def columns_like(df, names):
    # return columns that match any token exactly
    return [c for c in df.columns if c in names]

qcp_cols = columns_like(merged, feature_hints["qcp_encoding"] + feature_hints["qcp_retrieval"] + feature_hints["qcp_behavioral"])
moca_cols = columns_like(merged, feature_hints["moca_like"])

# There may be duplicates across suffixes; deduplicate while preserving order
def dedup(seq): 
    seen=set(); out=[]
    for x in seq:
        if x not in seen:
            out.append(x); seen.add(x)
    return out

qcp_cols  = dedup(qcp_cols)
moca_cols = dedup(moca_cols)

# If your CSVs have suffixes (e.g., N1_embic), catch those by regex:
def regex_pick(df, patterns):
    hits=[]
    for pat in patterns:
        hits += [c for c in df.columns if re.fullmatch(pat, c)]
    return dedup(hits)

if not qcp_cols:
    # accept suffixed names like N1|N1_embic
    qcp_cols = regex_pick(merged, [r"(N[1-4])(_.*)?", r"(R[1-3])(_.*)?", r"(M[1-9])(_.*)?"])
if not moca_cols:
    # accept common MoCA names with optional suffixes
    moca_cols = regex_pick(
        merged,
        [
            r"(TRAILS|CUBE|CLOCKCON|CLOCKNO|CLOCKHAN)(_.*)?",
            r"(LION|RHINO|CAMEL)(_.*)?",
            r"(IMMT1W[1-5]|IMMT2W[1-5])(_.*)?",
            r"(DIGFOR|DIGBACK|LETTERS|SERIAL[1-5]|REPEAT[12]|FFLUENCY|ABSTRAN|ABSMEAS)(_.*)?",
            r"(DELW[1-5]|DATE|MONTH|YEAR|DAY|PLACE|CITY|MOCA)(_.*)?",
        ]
    )

# -------------------------
# 6) Assemble X, y with explainable rationale
# -------------------------
meta_to_drop = [
    "RID","VISCODE","VISCODE2","EXAMDATE","PHASE","PTID","VISDATE",
    "USERDATE","USERDATE2","DD_CRF_VERSION_LABEL","LANGUAGE_CODE","HAS_QC_ERROR",
    "SOURCE","ID","SITEID","update_stamp","update_stamp_x","update_stamp_y"
]
meta_present = [c for c in meta_to_drop if c in merged.columns]
label_series = merged[label_col]

# Normalize labels into {CN,MCI,AD} if they are string-like; or map common ADNI numeric codes
def normalize_labels(y):
    # Strip whitespace/upper
    if y.dtype == object:
        y_norm = y.astype(str).str.upper().str.strip()
        # collapse several variants
        y_norm = y_norm.replace({
            "COGNITIVELY NORMAL":"CN", "NORMAL":"CN",
            "MILD COGNITIVE IMPAIRMENT":"MCI",
            "ALZHEIMER'S DISEASE":"AD", "ALZHEIMERS DISEASE":"AD", "ALZHEIMER DISEASE":"AD"
        })
        return y_norm
    else:
        # numeric coding (common DXCHANGE style): 1=CN, 2=MCI, 3=AD
        mapping = {1:"CN", 2:"MCI", 3:"AD"}
        return y.map(mapping)

y_str = normalize_labels(label_series)
valid_mask = y_str.isin(["CN","MCI","AD"])
merged = merged.loc[valid_mask].copy()
y_str = y_str.loc[valid_mask]

# Build X from numeric columns only, focusing on the hinted families
candidate_cols = dedup(qcp_cols + moca_cols)
# If none detected (rare), fallback to all numeric test-like columns
if not candidate_cols:
    # keep all numeric, then drop obvious metadata
    numeric_cols = merged.select_dtypes(include=[np.number]).columns.tolist()
    candidate_cols = [c for c in numeric_cols if c not in meta_present]

X_all = merged[candidate_cols].copy()
# Keep numeric only
X_all = X_all.select_dtypes(include=[np.number])
# Drop all-empty columns
X_all = X_all.dropna(axis=1, how="all")
# Drop rows with any missing values in X (simple baseline; you can swap with imputation)
mask_complete = X_all.notna().all(axis=1)
X = X_all.loc[mask_complete].copy()
y = y_str.loc[mask_complete].map({"CN":0,"MCI":1,"AD":2})

# If class imbalance is severe, warn:
print("[INFO] Class distribution:", dict(pd.Series(y).value_counts()))

# -------------------------
# 7) Train/Test + Model
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    class_weight="balanced_subsample"
)
model.fit(X_train, y_train)

print("\nClassification report (CN/MCI/AD):\n")
print(classification_report(y_test, model.predict(X_test), target_names=["CN","MCI","AD"]))

# -------------------------
# 8) Explainability with SHAP
# -------------------------
# Use TreeExplainer for forests
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Save top features (mean |SHAP|)
mean_abs = np.mean(np.abs(shap_values), axis=1) if isinstance(shap_values, list) else np.mean(np.abs(shap_values), axis=0)
# For multiclass, shap_values is list of arrays per class; combine:
if isinstance(shap_values, list):
    combined = np.mean(np.stack([np.abs(v) for v in shap_values], axis=0), axis=0)  # (n_samples, n_features)
    importances = combined.mean(axis=0)
else:
    importances = np.mean(np.abs(shap_values), axis=0)

feat_importance = (
    pd.DataFrame({"feature": X_test.columns, "mean_abs_shap": importances})
      .sort_values("mean_abs_shap", ascending=False)
)
os.makedirs("xai_outputs", exist_ok=True)
feat_importance.to_csv("xai_outputs/feature_importance_shap.csv", index=False)

# Plots
plt.figure()
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("xai_outputs/shap_summary_bar.png", dpi=200)
plt.close()

plt.figure()
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.savefig("xai_outputs/shap_beeswarm.png", dpi=200)
plt.close()

# Per-feature directionality (top 10)
top10 = feat_importance.head(10)["feature"].tolist()
per_feat_dir = {}
for f in top10:
    # correlation between feature value and model output (approx, using SHAP sign on class 'AD')
    try:
        ad_idx = 2  # class index for AD
        sh = shap_values[ad_idx] if isinstance(shap_values, list) else shap_values
        per_feat_dir[f] = float(np.corrcoef(X_test[f].values, sh[:, X_test.columns.get_loc(f)])[0,1])
    except Exception:
        per_feat_dir[f] = np.nan

with open("xai_outputs/top10_directionality.json","w") as f:
    json.dump(per_feat_dir, f, indent=2)

print("\n[OK] Saved:")
print(" - xai_outputs/feature_importance_shap.csv")
print(" - xai_outputs/shap_summary_bar.png")
print(" - xai_outputs/shap_beeswarm.png")
print(" - xai_outputs/top10_directionality.json")

# -------------------------
# 9) Human-readable rationale
# -------------------------
def rationale_for(feature):
    f = feature.upper()
    # qCP families
    if re.fullmatch(r"N[1-4](_.*)?", f):
        return "Encoding efficiency (earlier repetitions). Lower values → weaker initial learning."
    if re.fullmatch(r"R[1-3](_.*)?", f):
        return "Retrieval dynamics (immediate/delayed recall). Lower values → retrieval difficulty."
    if re.fullmatch(r"M[1-9](_.*)?", f):
        return "Behavioral/recognition memory parameter. Lower values → poorer discrimination or recall rate."
    # MoCA families
    if f.startswith("TRAILS") or f.startswith("CUBE") or f.startswith("CLOCK"):
        return "Visuospatial / executive (MoCA). Sensitive to early executive/visuospatial decline."
    if f in {"LION","RHINO","CAMEL"}:
        return "Naming (MoCA). Lexical retrieval; declines with semantic memory issues."
    if f.startswith("IMMT") or f.startswith("DELW"):
        return "Immediate/delayed word recall (MoCA). Episodic memory (hippocampal)."
    if f in {"DIGFOR","DIGBACK","LETTERS"} or f.startswith("SERIAL"):
        return "Attention / working memory (MoCA)."
    if f in {"REPEAT1","REPEAT2","FFLUENCY","ABSTRAN","ABSMEAS"}:
        return "Language / abstraction (MoCA)."
    if f in {"DATE","MONTH","YEAR","DAY","PLACE","CITY"}:
        return "Orientation (MoCA). Later-stage sensitivity."
    if f == "MOCA" or f.startswith("MOCA_"):
        return "Global MoCA score (composite)."
    return "Model-selected feature; inspect SHAP for direction and magnitude."

explain_rows = []
for _, row in feat_importance.head(20).iterrows():
    explain_rows.append({
        "feature": row["feature"],
        "mean_abs_shap": float(row["mean_abs_shap"]),
        "why_important": rationale_for(row["feature"])
    })
pd.DataFrame(explain_rows).to_csv("xai_outputs/top20_features_explained.csv", index=False)
print(" - xai_outputs/top20_features_explained.csv")


[WARN] Could not read PDF (docs/ADAS_EMBIC_qCP_Methods.pdf): [Errno 2] No such file or directory: 'docs/ADAS_EMBIC_qCP_Methods.pdf'
[INFO] Feature hints inferred from PDF:
{
  "qcp_encoding": [
    "N1",
    "N2",
    "N3",
    "N4"
  ],
  "qcp_retrieval": [
    "R1",
    "R2",
    "R3"
  ],
  "qcp_behavioral": [
    "M1",
    "M2",
    "M3"
  ],
  "moca_like": [
    "TRAILS",
    "CUBE",
    "CLOCKCON",
    "CLOCKNO",
    "CLOCKHAN",
    "LION",
    "RHINO",
    "CAMEL",
    "IMMT1W1",
    "IMMT1W2",
    "IMMT1W3",
    "IMMT1W4",
    "IMMT1W5",
    "IMMT2W1",
    "IMMT2W2",
    "IMMT2W3",
    "IMMT2W4",
    "IMMT2W5",
    "DIGFOR",
    "DIGBACK",
    "LETTERS",
    "SERIAL1",
    "SERIAL2",
    "SERIAL3",
    "SERIAL4",
    "SERIAL5",
    "REPEAT1",
    "REPEAT2",
    "FFLUENCY",
    "ABSTRAN",
    "ABSMEAS",
    "DELW1",
    "DELW2",
    "DELW3",
    "DELW4",
    "DELW5",
    "DATE",
    "MONTH",
    "YEAR",
    "DAY",
    "PLACE",
    "CITY",
    "MOCA"
  ],
  "adas_like": [
    "AD

ValueError: Per-column arrays must each be 1-dimensional

In [8]:
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# -----------------------------
# 1. Load data
# -----------------------------
embic = pd.read_csv("Assessments/ADNI_EMBICDCB_08Jul2025.csv")
moca = pd.read_csv("Assessments/MOCA_08Jul2025.csv")
dxsum = pd.read_csv("Assessments/DXSUM_08Jul2025.csv")

# Example join on RID
data = embic.merge(moca, on="RID", how="inner")
data = data.merge(dxsum, on="RID", how="inner")

# Diagnosis mapping
diagnosis_map = {"CN": 0, "MCI": 1, "AD": 2}
data["Diagnosis_num"] = data["DIAGNOSIS"].map(diagnosis_map)

# -----------------------------
# 2. Define feature groups
# -----------------------------
feature_groups = {
    "qcp_encoding": ["N1","N2","N3","N4"],
    "qcp_retrieval": ["R1","R2","R3"],
    "qcp_behavioral": ["M1","M2","M3"],
    "moca_like": [
        "TRAILS","CUBE","CLOCKCON","CLOCKNO","CLOCKHAN",
        "LION","RHINO","CAMEL",
        "IMMT1W1","IMMT1W2","IMMT1W3","IMMT1W4","IMMT1W5",
        "IMMT2W1","IMMT2W2","IMMT2W3","IMMT2W4","IMMT2W5",
        "DIGFOR","DIGBACK","LETTERS",
        "SERIAL1","SERIAL2","SERIAL3","SERIAL4","SERIAL5",
        "REPEAT1","REPEAT2","FFLUENCY","ABSTRAN","ABSMEAS",
        "DELW1","DELW2","DELW3","DELW4","DELW5",
        "DATE","MONTH","YEAR","DAY","PLACE","CITY","MOCA"
    ],
    "adas_like": ["ADAS","WORD","RECALL","RECOGNITION"]
}

# Keep only features that actually exist in data
available_features = [f for f in sum(feature_groups.values(), []) if f in data.columns]

print("Available features used for training:", available_features)

# -----------------------------
# 3. Prepare X, y
# -----------------------------
X = data[available_features].fillna(0)
y = data["Diagnosis_num"]

# -----------------------------
# 4. Train/Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# 5. Train Model
# -----------------------------
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# -----------------------------
# 6. Evaluate
# -----------------------------
y_pred = model.predict(X_test)
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["CN", "MCI", "AD"]))

# -----------------------------
# 7. Explain with SHAP
# -----------------------------
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Global feature importance
shap.summary_plot(shap_values, X_test, plot_type="bar")

# Detailed feature impact
shap.summary_plot(shap_values, X_test)


Available features used for training: ['N1', 'N2', 'N3', 'N4', 'R1', 'R2', 'R3', 'M1', 'M2', 'M3', 'TRAILS', 'CUBE', 'CLOCKCON', 'CLOCKNO', 'CLOCKHAN', 'LION', 'RHINO', 'CAMEL', 'IMMT1W1', 'IMMT1W2', 'IMMT1W3', 'IMMT1W4', 'IMMT1W5', 'IMMT2W1', 'IMMT2W2', 'IMMT2W3', 'IMMT2W4', 'IMMT2W5', 'DIGFOR', 'DIGBACK', 'LETTERS', 'SERIAL1', 'SERIAL2', 'SERIAL3', 'SERIAL4', 'SERIAL5', 'REPEAT1', 'REPEAT2', 'FFLUENCY', 'ABSTRAN', 'ABSMEAS', 'DELW1', 'DELW2', 'DELW3', 'DELW4', 'DELW5', 'DATE', 'MONTH', 'YEAR', 'DAY', 'PLACE', 'CITY', 'MOCA']


ValueError: Input y contains NaN.

In [10]:
# explainable_pipeline_adni.py
# Run in the environment where you unzipped Assessments.zip
# Requires: pandas, numpy, scikit-learn, shap, matplotlib
# pip install pandas numpy scikit-learn shap matplotlib

import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
import shap
import matplotlib.pyplot as plt

# ---------------------------
# CONFIG: paths (adjust if needed)
# ---------------------------
POSSIBLE_DIRS = [
    "/mnt/data/Assessments_unzipped",  # where this session previously extracted files
    "Assessments",                      # alternate relative path (used earlier)
    "/mnt/data/Assessments"
]
DATA_DIR = next((d for d in POSSIBLE_DIRS if os.path.isdir(d)), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find the Assessments folder. Update POSSIBLE_DIRS in the script.")

# expected filenames (from your zip)
EMBIC_PAT = os.path.join(DATA_DIR, "ADNI_EMBICDCB_08Jul2025.csv")
MOCA_PAT  = os.path.join(DATA_DIR, "MOCA_08Jul2025.csv")
DXSUM_PAT = os.path.join(DATA_DIR, "DXSUM_08Jul2025.csv")

for p in (EMBIC_PAT, MOCA_PAT, DXSUM_PAT):
    if not os.path.isfile(p):
        raise FileNotFoundError(f"Required file not found: {p}")

print("Data dir:", DATA_DIR)
print("EMBIC:", EMBIC_PAT)
print("MOCA :", MOCA_PAT)
print("DXSUM:", DXSUM_PAT)

# ---------------------------
# 1) Load files
# ---------------------------
embic = pd.read_csv(EMBIC_PAT)
moca  = pd.read_csv(MOCA_PAT)
dxsum = pd.read_csv(DXSUM_PAT)

# ---------------------------
# 2) Merge on common keys
# Try RID + VISCODE2 primarily, fallback to RID only if necessary
# ---------------------------
def find_join_keys(df1, df2):
    # prefer RID + VISCODE2
    if "RID" in df1.columns and "RID" in df2.columns and "VISCODE2" in df1.columns and "VISCODE2" in df2.columns:
        return ["RID","VISCODE2"], ["RID","VISCODE2"]
    # fallback: RID + VISCODE
    if "RID" in df1.columns and "RID" in df2.columns and "VISCODE" in df1.columns and "VISCODE" in df2.columns:
        return ["RID","VISCODE"], ["RID","VISCODE"]
    # fallback: RID only
    if "RID" in df1.columns and "RID" in df2.columns:
        return ["RID"], ["RID"]
    raise KeyError("No suitable merge keys found between datasets (expected RID and VISCODE2/VISCODE).")

k_embic, k_moca = find_join_keys(embic, moca)
merged = pd.merge(embic, moca, left_on=k_embic, right_on=k_moca, how="inner", suffixes=("_embic","_moca"))
print("Merged EMBIC+MOCA shape:", merged.shape)

# Merge diagnosis
# For DXSUM prefer DXNORM, DXMCI, DXAD or DIAGNOSIS; we'll merge on same key set
k_dxsum, k_dxsum_right = find_join_keys(dxsum, merged)  # dxsum left side keys
merged = pd.merge(merged, dxsum, left_on=k_embic, right_on=k_dxsum, how="inner", suffixes=("","_dx"))
print("After merging DXSUM shape:", merged.shape)

# ---------------------------
# 3) Build label (Diagnosis)
# DXSUM in your folder contains DXNORM, DXMCI, DXAD columns (1/0), so prefer those.
# Fallback: try a DIAGNOSIS text column
# ---------------------------
if "DXAD" in merged.columns and "DXMCI" in merged.columns and "DXNORM" in merged.columns:
    def diag_from_flags(row):
        if row.get("DXAD") == 1: return "AD"
        if row.get("DXMCI") == 1: return "MCI"
        if row.get("DXNORM") == 1: return "CN"
        return None
    merged["Diagnosis"] = merged.apply(diag_from_flags, axis=1)
elif "DIAGNOSIS" in merged.columns:
    merged["Diagnosis"] = merged["DIAGNOSIS"].astype(str).str.upper().str.strip().map(
        {"COGNITIVELY NORMAL":"CN","NORMAL":"CN","CN":"CN","MCI":"MCI","AD":"AD",
         "ALZHEIMER'S DISEASE":"AD","ALZHEIMERS DISEASE":"AD"}
    )
else:
    # Try to find first column starting with 'DX' with categorical values
    dx_like = [c for c in merged.columns if c.upper().startswith("DX")]
    if dx_like:
        merged["Diagnosis"] = merged[dx_like[0]].astype(str)
    else:
        raise KeyError("Could not find diagnosis columns in merged DXSUM. Inspect DXSUM file.")

# Keep only rows with CN/MCI/AD
merged = merged[merged["Diagnosis"].isin(["CN","MCI","AD"])].copy()
print("Rows with valid Diagnosis:", merged.shape[0])
if merged.shape[0] == 0:
    raise ValueError("No rows with CN/MCI/AD labels found after merging. Check DXSUM alignment.")

# ---------------------------
# 4) Select feature columns robustly
# Use qCP families (N1..N4, R1..R3, M1..), and MoCA subtests if present
# ---------------------------
qcp_encoding = ["N1","N2","N3","N4"]
qcp_retrieval = ["R1","R2","R3"]
qcp_behavioral = ["M1","M2","M3"]  # EMBIC uses M1..M3 in csv
moca_candidates = [
    "TRAILS","CUBE","CLOCKCON","CLOCKNO","CLOCKHAN",
    "LION","RHINO","CAMEL",
    "IMMT1W1","IMMT1W2","IMMT1W3","IMMT1W4","IMMT1W5",
    "IMMT2W1","IMMT2W2","IMMT2W3","IMMT2W4","IMMT2W5",
    "DIGFOR","DIGBACK","LETTERS",
    "SERIAL1","SERIAL2","SERIAL3","SERIAL4","SERIAL5",
    "REPEAT1","REPEAT2","FFLUENCY","ABSTRAN","ABSMEAS",
    "DELW1","DELW2","DELW3","DELW4","DELW5",
    "DATE","MONTH","YEAR","DAY","PLACE","CITY","MOCA"
]

# Build candidate list but only keep those present in merged.columns
candidate_features = []
for cand in qcp_encoding + qcp_retrieval + qcp_behavioral + moca_candidates:
    if cand in merged.columns:
        candidate_features.append(cand)

# Also include M1..M* if extra M columns exist (safe detection)
extra_M = [c for c in merged.columns if c.startswith("M") and c not in candidate_features and len(c)<=4]
for c in extra_M:
    candidate_features.append(c)

if not candidate_features:
    raise ValueError("No candidate features found in merged data. Inspect column names.")

print("Selected features (count={}):".format(len(candidate_features)))
print(candidate_features)

# ---------------------------
# 5) Prepare X and y
# ---------------------------
X = merged[candidate_features].copy()
y = merged["Diagnosis"].map({"CN":0,"MCI":1,"AD":2}).astype(int)

# Keep only numeric columns in X (drop any stray strings)
X = X.select_dtypes(include=[np.number])

# Impute missing numeric values (median)
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

# Drop columns that remain constant or all zero after imputation
nunique = X_imputed.nunique()
const_cols = nunique[nunique <= 1].index.tolist()
if const_cols:
    print("Dropping constant columns:", const_cols)
    X_imputed.drop(columns=const_cols, inplace=True)

# Align y with X rows (in case any rows dropped)
X = X_imputed.copy()
y = y.loc[X.index]

print("Final X shape:", X.shape, "y shape:", y.shape)
print("Class distribution:", dict(pd.Series(y).value_counts()))

# ---------------------------
# 6) Train/test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ---------------------------
# 7) Train RandomForest (balanced)
# ---------------------------
model = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced_subsample", n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["CN","MCI","AD"]))

# ---------------------------
# 8) SHAP explainability (global + per-group)
# ---------------------------
os.makedirs("/mnt/data/xai_outputs", exist_ok=True)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Compute mean absolute SHAP across classes for global ranking
if isinstance(shap_values, list):
    # shap_values is a list per class: shape each = (n_samples, n_features)
    combined = np.mean(np.abs(np.stack(shap_values, axis=0)), axis=(0,1))  # mean over classes & samples
    # But better compute per-feature mean abs across samples across classes:
    importances = np.mean(np.mean(np.abs(np.stack(shap_values, axis=0)), axis=2), axis=0)
else:
    importances = np.mean(np.abs(shap_values), axis=0)

feat_imp = pd.DataFrame({"feature": X_test.columns, "mean_abs_shap": importances})
feat_imp = feat_imp.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
feat_imp.to_csv("/mnt/data/xai_outputs/feature_importance_shap.csv", index=False)
print("Saved feature importance CSV to /mnt/data/xai_outputs/feature_importance_shap.csv")

# SHAP summary plots
plt.figure(figsize=(8,6))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("/mnt/data/xai_outputs/shap_summary_bar.png", dpi=200)
plt.close()

plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.savefig("/mnt/data/xai_outputs/shap_beeswarm.png", dpi=200)
plt.close()

# Save top 20 with human friendly rationale
def rationale_for(feature):
    f = feature.upper()
    if f.startswith("N"):
        return "Encoding efficiency (qCP N1..N4). Lower → poorer encoding."
    if f.startswith("R"):
        return "Retrieval dynamics (qCP R1..R3). Lower → retrieval difficulty."
    if f.startswith("M"):
        return "Behavioral memory/recognition param (M*)."
    if f in {"TRAILS","CUBE"} or f.startswith("CLOCK"):
        return "Visuospatial / executive function (MoCA)."
    if f in {"LION","RHINO","CAMEL"}:
        return "Naming (semantic memory)."
    if f.startswith("IMMT") or f.startswith("DELW"):
        return "Immediate/delayed recall (MoCA memory)."
    if f in {"DIGFOR","DIGBACK","LETTERS"} or f.startswith("SERIAL"):
        return "Attention / working memory."
    if f == "MOCA":
        return "Global MoCA score."
    return "Model-selected numeric feature."

top = feat_imp.head(20).copy()
top["why"] = top["feature"].apply(rationale_for)
top.to_csv("/mnt/data/xai_outputs/top20_features_explained.csv", index=False)
print("Saved top-20 explained features to /mnt/data/xai_outputs/top20_features_explained.csv")

# Save confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=["CN","MCI","AD"], columns=["CN","MCI","AD"])
cm_df.to_csv("/mnt/data/xai_outputs/confusion_matrix.csv")
print("Saved confusion matrix to /mnt/data/xai_outputs/confusion_matrix.csv")

print("\nOutputs in /mnt/data/xai_outputs/:")
for f in os.listdir("/mnt/data/xai_outputs"):
    print(" -", f)

# Example: print top 10 features
print("\nTop features (by SHAP):")
print(top.head(10).to_string(index=False))


Data dir: Assessments
EMBIC: Assessments\ADNI_EMBICDCB_08Jul2025.csv
MOCA : Assessments\MOCA_08Jul2025.csv
DXSUM: Assessments\DXSUM_08Jul2025.csv
Merged EMBIC+MOCA shape: (7052, 69)
After merging DXSUM shape: (7036, 108)
Rows with valid Diagnosis: 0


ValueError: No rows with CN/MCI/AD labels found after merging. Check DXSUM alignment.

In [13]:
def extract_diagnosis(dxsum):
    """
    Create a single Diagnosis column from ADNI DXSUM flags.
    Priority order: AD > MCI > CN
    """
    diagnosis = []

    for _, row in dxsum.iterrows():
        if "DXMCI" in row and row["DXMCI"] == 1.0:
            diagnosis.append("MCI")
        elif "DXNORM" in row and row["DXNORM"] == 1.0:
            diagnosis.append("CN")
        elif "DXOTHDEM" in row and row["DXOTHDEM"] == 1.0:  # dementia/AD flag
            diagnosis.append("AD")
        else:
            diagnosis.append(None)  # no valid diagnosis

    dxsum["Diagnosis"] = diagnosis
    return dxsum


# Apply to your dataframe
dxsum = extract_diagnosis(dxsum)

print(dxsum["Diagnosis"].value_counts(dropna=False))


Diagnosis
None    12679
MCI      1601
CN       1130
AD          8
Name: count, dtype: int64


In [12]:
print(dxsum.filter(like="DX").head(10))
print(dxsum["DXCHANGE"].value_counts())


   DXNORM  DXNODEP  DXMCI DXMDES  DXMPTR1  DXMPTR2  DXMPTR3  DXMPTR4  DXMPTR5  \
0     1.0     -4.0   -4.0     -4     -4.0     -4.0     -4.0     -4.0     -4.0   
1    -4.0     -4.0   -4.0     -4     -4.0     -4.0     -4.0     -4.0     -4.0   
2     1.0     -4.0   -4.0     -4     -4.0     -4.0     -4.0     -4.0     -4.0   
3     1.0     -4.0   -4.0     -4     -4.0     -4.0     -4.0     -4.0     -4.0   
4    -4.0     -4.0   -4.0     -4     -4.0     -4.0     -4.0     -4.0     -4.0   
5     1.0     -4.0   -4.0     -4     -4.0     -4.0     -4.0     -4.0     -4.0   
6    -4.0     -4.0    1.0      1      1.0      1.0      1.0      1.0      1.0   
7     1.0     -4.0   -4.0     -4     -4.0     -4.0     -4.0     -4.0     -4.0   
8     1.0     -4.0   -4.0     -4     -4.0     -4.0     -4.0     -4.0     -4.0   
9     1.0     -4.0   -4.0     -4     -4.0     -4.0     -4.0     -4.0     -4.0   

   DXMPTR6  ...  DXAPROB DXAPOSS  DXPARK  DXPDES  DXPCOG  DXPATYP DXDEP  \
0     -4.0  ...       -4      -4 

KeyError: 'DXCHANGE'

In [14]:
merged = embic_moca.merge(dxsum[["RID", "Diagnosis"]], on="RID", how="inner")
print("After merging DXSUM shape:", merged.shape)
print("Diagnosis counts:", merged["Diagnosis"].value_counts(dropna=False))


NameError: name 'embic_moca' is not defined

In [18]:
# explainable_pipeline_adni_fixed.py
# Run where you unzipped Assessments.zip
# Requirements: pandas, numpy, scikit-learn, shap, matplotlib
# pip install pandas numpy scikit-learn shap matplotlib

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
import shap
import matplotlib.pyplot as plt

# ---------------------------
# CONFIG: possible locations (adjust if needed)
# ---------------------------
POSSIBLE_DIRS = [
    "/mnt/data/Assessments_unzipped",
    "Assessments",
    "/mnt/data/Assessments"
]
DATA_DIR = next((d for d in POSSIBLE_DIRS if os.path.isdir(d)), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find Assessments folder. Update POSSIBLE_DIRS in the script.")

EMBIC_PATH = os.path.join(DATA_DIR, "ADNI_EMBICDCB_08Jul2025.csv")
MOCA_PATH  = os.path.join(DATA_DIR, "MOCA_08Jul2025.csv")
DXSUM_PATH = os.path.join(DATA_DIR, "DXSUM_08Jul2025.csv")

for p in (EMBIC_PATH, MOCA_PATH, DXSUM_PATH):
    if not os.path.isfile(p):
        raise FileNotFoundError(f"Required file not found: {p}")

print("Data dir:", DATA_DIR)
print("EMBIC:", EMBIC_PATH)
print("MOCA :", MOCA_PATH)
print("DXSUM:", DXSUM_PATH)

# ---------------------------
# 1) Load CSVs
# ---------------------------
embic = pd.read_csv(EMBIC_PATH)
moca  = pd.read_csv(MOCA_PATH)
dxsum = pd.read_csv(DXSUM_PATH)

# Quick column check (helpful debug)
print("\nSample columns (EMBIC):", embic.columns.tolist()[:20])
print("Sample columns (MOCA):", moca.columns.tolist()[:20])
print("Sample columns (DXSUM):", dxsum.columns.tolist()[:20])

# ---------------------------
# 2) Merge EMBIC + MOCA
# Prefer RID + VISCODE2, fallback to RID + VISCODE, then RID only
# ---------------------------
def pick_keys(a, b):
    if {"RID","VISCODE2"}.issubset(a.columns) and {"RID","VISCODE2"}.issubset(b.columns):
        return ["RID","VISCODE2"], ["RID","VISCODE2"]
    if {"RID","VISCODE"}.issubset(a.columns) and {"RID","VISCODE"}.issubset(b.columns):
        return ["RID","VISCODE"], ["RID","VISCODE"]
    if "RID" in a.columns and "RID" in b.columns:
        return ["RID"], ["RID"]
    raise KeyError("No suitable join keys (expect RID and VISCODE/VISCODE2)")

k_embic, k_moca = pick_keys(embic, moca)
merged = pd.merge(embic, moca, left_on=k_embic, right_on=k_moca, how="inner", suffixes=("_embic","_moca"))
print("\nMerged EMBIC+MOCA shape:", merged.shape)

# ---------------------------
# 3) Merge DXSUM (diagnosis info)
# Use same join keys where possible (try VISCODE2 then VISCODE then RID)
# ---------------------------
# determine best keys between dxsum and merged
if {"RID","VISCODE2"}.issubset(dxsum.columns) and {"RID","VISCODE2"}.issubset(merged.columns):
    k_dxsum, k_merged = ["RID","VISCODE2"], ["RID","VISCODE2"]
elif {"RID","VISCODE"}.issubset(dxsum.columns) and {"RID","VISCODE"}.issubset(merged.columns):
    k_dxsum, k_merged = ["RID","VISCODE"], ["RID","VISCODE"]
elif "RID" in dxsum.columns and "RID" in merged.columns:
    k_dxsum, k_merged = ["RID"], ["RID"]
else:
    raise KeyError("No common merge keys between DXSUM and merged dataframes")

merged = pd.merge(merged, dxsum, left_on=k_merged, right_on=k_dxsum, how="inner", suffixes=("","_dx"))
print("After merging DXSUM shape:", merged.shape)

# ---------------------------
# 4) Build Diagnosis label from DXSUM flags
# ADNI often encodes diagnosis as binary flags (DXNORM, DXMCI, DXMDES, DXOTHDEM, DXAD etc.)
# We'll scan dx columns and map them robustly.
# ---------------------------
# find all dx-like columns (case-insensitive)
dx_columns = [c for c in merged.columns if c.upper().startswith("DX")]
print("\nDX-like columns found in merged:", dx_columns)

# Convert candidate dx columns to numeric (coerce non-numeric)
for c in dx_columns:
    merged[c] = pd.to_numeric(merged[c], errors="coerce")

# Identify which columns likely indicate each diagnosis by name heuristics
norm_cols = [c for c in dx_columns if "NORM" in c.upper()]
mci_cols  = [c for c in dx_columns if "MCI" in c.upper()]
ad_cols   = [c for c in dx_columns if ("AD" in c.upper() and "MD" not in c.upper()) or "DEM" in c.upper() or "OTHDEM" in c.upper() or "ALZ" in c.upper()]

print("Candidate norm_cols:", norm_cols)
print("Candidate mci_cols :", mci_cols)
print("Candidate ad_cols  :", ad_cols)

# If no AD columns by heuristic, try exact DXAD
if not ad_cols:
    ad_cols = [c for c in dx_columns if c.upper().startswith("DXAD") or c.upper().endswith("AD")]
    print("Fallback ad_cols:", ad_cols)

# Build row-wise diagnosis (AD priority, then MCI, then CN)
def infer_diag_row(row):
    # AD if any ad_cols has positive/1
    for c in ad_cols:
        val = row.get(c)
        if pd.notna(val) and float(val) == 1.0:
            return "AD"
    # MCI
    for c in mci_cols:
        val = row.get(c)
        if pd.notna(val) and float(val) == 1.0:
            return "MCI"
    # Normal
    for c in norm_cols:
        val = row.get(c)
        if pd.notna(val) and float(val) == 1.0:
            return "CN"
    return None

merged["Diagnosis"] = merged.apply(infer_diag_row, axis=1)

print("\nDiagnosis value counts (raw):")
print(merged["Diagnosis"].value_counts(dropna=False))

# fallback assignment if many None (try direct DIAGNOSIS text column or numeric flags)
if merged["Diagnosis"].isna().sum() > 0:
    if "DIAGNOSIS" in merged.columns:
        merged["Diagnosis"] = merged["DIAGNOSIS"].astype(str).str.upper().str.strip().replace({
            "COGNITIVELY NORMAL":"CN","NORMAL":"CN","CN":"CN",
            "MCI":"MCI","AD":"AD","ALZHEIMER'S DISEASE":"AD","ALZHEIMERS DISEASE":"AD"
        })
    else:
        # attempt brute-force assignment from dx_columns
        for c in dx_columns:
            name = c.upper()
            try:
                if "MCI" in name:
                    merged.loc[merged[c] == 1, "Diagnosis"] = "MCI"
                if "NORM" in name:
                    merged.loc[merged[c] == 1, "Diagnosis"] = "CN"
                if "AD" in name or "DEM" in name or "ALZ" in name or "OTHDEM" in name:
                    merged.loc[merged[c] == 1, "Diagnosis"] = "AD"
            except Exception:
                pass
    print("\nDiagnosis after fallback assignment:")
    print(merged["Diagnosis"].value_counts(dropna=False))

# Keep only rows with valid labels
merged = merged[merged["Diagnosis"].isin(["CN","MCI","AD"])].copy()
print("\nRows with valid Diagnosis:", merged.shape[0])
if merged.shape[0] == 0:
    raise ValueError("No rows with CN/MCI/AD labels found after merging. Check DXSUM alignment and flag values.")

# ---------------------------
# 5) Select features (qCP + MoCA subscores)
# ---------------------------
qcp_encoding = ["N1","N2","N3","N4"]
qcp_retrieval = ["R1","R2","R3"]
qcp_behavioral = ["M1","M2","M3"]
moca_candidates = [
    "TRAILS","CUBE","CLOCKCON","CLOCKNO","CLOCKHAN",
    "LION","RHINO","CAMEL",
    "IMMT1W1","IMMT1W2","IMMT1W3","IMMT1W4","IMMT1W5",
    "IMMT2W1","IMMT2W2","IMMT2W3","IMMT2W4","IMMT2W5",
    "DIGFOR","DIGBACK","LETTERS",
    "SERIAL1","SERIAL2","SERIAL3","SERIAL4","SERIAL5",
    "REPEAT1","REPEAT2","FFLUENCY","ABSTRAN","ABSMEAS",
    "DELW1","DELW2","DELW3","DELW4","DELW5",
    "DATE","MONTH","YEAR","DAY","PLACE","CITY","MOCA"
]

candidate_features = []
for cand in qcp_encoding + qcp_retrieval + qcp_behavioral + moca_candidates:
    if cand in merged.columns:
        candidate_features.append(cand)

# Add extra M* if present
extra_M = [c for c in merged.columns if c.startswith("M") and c not in candidate_features and len(c) <= 5]
for c in extra_M:
    candidate_features.append(c)

if not candidate_features:
    raise ValueError("No candidate features found. Inspect merged.columns to adapt feature list.")

print("\nSelected features ({}):".format(len(candidate_features)))
print(candidate_features)

# ---------------------------
# 6) Prepare X and y, numeric conversion + impute
# ---------------------------
X = merged[candidate_features].copy()
X = X.apply(pd.to_numeric, errors="coerce")
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

# drop constant columns
const_cols = [c for c in X_imputed.columns if X_imputed[c].nunique() <= 1]
if const_cols:
    print("Dropping constant columns:", const_cols)
    X_imputed.drop(columns=const_cols, inplace=True)

X = X_imputed.copy()
y = merged["Diagnosis"].map({"CN":0,"MCI":1,"AD":2}).astype(int)
y = y.loc[X.index]

print("\nFinal X shape:", X.shape, "y shape:", y.shape)
print("Class distribution:", dict(pd.Series(y).value_counts()))

# ---------------------------
# 7) Train/test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# ---------------------------
# 8) Train RandomForest (balanced)
# ---------------------------
model = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced_subsample", n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["CN","MCI","AD"]))

# ---------------------------
# 9) SHAP explainability and outputs
# ---------------------------
out_dir = "/mnt/data/xai_outputs"
os.makedirs(out_dir, exist_ok=True)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Compute mean absolute importance across classes
if isinstance(shap_values, list):
    importances = np.mean(np.mean(np.abs(np.stack(shap_values, axis=0)), axis=1), axis=0)
else:
    importances = np.mean(np.abs(shap_values), axis=0)

feat_imp = pd.DataFrame({"feature": X_test.columns, "mean_abs_shap": importances})
feat_imp = feat_imp.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
feat_imp.to_csv(os.path.join(out_dir, "feature_importance_shap.csv"), index=False)

# plots
plt.figure(figsize=(8,6))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "shap_summary_bar.png"), dpi=200)
plt.close()

plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "shap_beeswarm.png"), dpi=200)
plt.close()

# top20 explained CSV with short rationale
def rationale_for(feature):
    f = feature.upper()
    if f.startswith("N"): return "Encoding efficiency (qCP). Lower → poorer encoding."
    if f.startswith("R"): return "Retrieval dynamics (qCP)."
    if f.startswith("M"): return "Behavioral/recognition memory param."
    if f in {"TRAILS","CUBE"} or f.startswith("CLOCK"): return "Visuospatial / executive (MoCA)."
    if f in {"LION","RHINO","CAMEL"}: return "Naming (MoCA)."
    if f.startswith("IMMT") or f.startswith("DELW"): return "Immediate/delayed recall (MoCA)."
    if f in {"DIGFOR","DIGBACK","LETTERS"} or f.startswith("SERIAL"): return "Attention / working memory."
    if f == "MOCA": return "Global MoCA score."
    return "Numeric cognitive feature."

top = feat_imp.head(20).copy()
top["why"] = top["feature"].apply(rationale_for)
top.to_csv(os.path.join(out_dir, "top20_features_explained.csv"), index=False)

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=["CN","MCI","AD"], columns=["CN","MCI","AD"])
cm_df.to_csv(os.path.join(out_dir, "confusion_matrix.csv"))

print("\nSaved XAI outputs to:", out_dir)
print("Top features:", top.head(10).to_string(index=False))


Data dir: Assessments
EMBIC: Assessments\ADNI_EMBICDCB_08Jul2025.csv
MOCA : Assessments\MOCA_08Jul2025.csv
DXSUM: Assessments\DXSUM_08Jul2025.csv

Sample columns (EMBIC): ['RID', 'VISCODE2', 'N1', 'N2', 'N3', 'N4', 'R1', 'R2', 'R3', 'M1', 'M2', 'M3', 'update_stamp']
Sample columns (MOCA): ['PHASE', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'VISDATE', 'TRAILS', 'CUBE', 'CLOCKCON', 'CLOCKNO', 'CLOCKHAN', 'LION', 'RHINO', 'CAMEL', 'IMMT1W1', 'IMMT1W2', 'IMMT1W3', 'IMMT1W4', 'IMMT1W5', 'IMMT2W1']
Sample columns (DXSUM): ['PHASE', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE', 'DIAGNOSIS', 'DXNORM', 'DXNODEP', 'DXMCI', 'DXMDES', 'DXMPTR1', 'DXMPTR2', 'DXMPTR3', 'DXMPTR4', 'DXMPTR5', 'DXMPTR6', 'DXMDUE', 'DXMOTHET', 'DXDSEV']

Merged EMBIC+MOCA shape: (7052, 69)
After merging DXSUM shape: (7036, 108)

DX-like columns found in merged: ['DXNORM', 'DXNODEP', 'DXMCI', 'DXMDES', 'DXMPTR1', 'DXMPTR2', 'DXMPTR3', 'DXMPTR4', 'DXMPTR5', 'DXMPTR6', 'DXMDUE', 'DXMOTHET', 'DXDSEV', 'DXDDUE', 'DXAD', 'DXA

ValueError: No rows with CN/MCI/AD labels found after merging. Check DXSUM alignment and flag values.

In [19]:
import pandas as pd

# ---------------------------
# 1) Load Data
# ---------------------------
embic = pd.read_csv("Assessments/ADNI_EMBICDCB_08Jul2025.csv")
moca  = pd.read_csv("Assessments/MOCA_08Jul2025.csv")
dxsum = pd.read_csv("Assessments/DXSUM_08Jul2025.csv")

print("Shapes -> EMBIC:", embic.shape, " MOCA:", moca.shape, " DXSUM:", dxsum.shape)


# ---------------------------
# 2) Merge (RID + VISCODE2 is the stable key across all)
# ---------------------------
merged = pd.merge(embic, moca, on=["RID","VISCODE2"], how="inner")
merged = pd.merge(merged, dxsum, on=["RID","VISCODE2"], how="inner")

print("After merging ->", merged.shape)


# ---------------------------
# 3) Assign Diagnosis
# ---------------------------
def assign_diagnosis(df):
    diag = pd.Series(index=df.index, dtype="object")

    # 1. Directly from DIAGNOSIS column (if usable)
    if "DIAGNOSIS" in df.columns:
        diag = df["DIAGNOSIS"].astype(str).str.upper().map({
            "CN": "CN",
            "NORMAL": "CN",
            "CONTROL": "CN",
            "MCI": "MCI",
            "AD": "AD",
            "DEMENTIA": "AD"
        })

    # 2. Fallback from DX flags (if DIAGNOSIS is missing/None)
    if diag.isna().all():
        if "DXNORM" in df.columns:
            diag[df["DXNORM"] == 1] = "CN"
        if "DXMCI" in df.columns:
            diag[df["DXMCI"] == 1] = "MCI"
        if "DXAD" in df.columns:
            diag[df["DXAD"] == 1] = "AD"
        if "DXOTHDEM" in df.columns:
            diag[df["DXOTHDEM"] == 1] = "AD"

    return diag

merged["Diagnosis"] = assign_diagnosis(merged)

print("\nDiagnosis value counts (raw):")
print(merged["Diagnosis"].value_counts(dropna=False))


# ---------------------------
# 4) Keep only CN / MCI / AD
# ---------------------------
merged = merged[merged["Diagnosis"].isin(["CN","MCI","AD"])]

print("\nRows with valid Diagnosis:", merged.shape[0])


# ---------------------------
# 5) Select Feature Sets
# ---------------------------
# qCP features (from EMBIC)
qcp_features = [c for c in merged.columns if c.startswith(("N","R","M"))]

# MoCA subscores (from MOCA)
moca_features = [
    "TRAILS","CUBE","CLOCKCON","CLOCKNO","CLOCKHAN",
    "LION","RHINO","CAMEL",
    "IMMT1W1","IMMT1W2","IMMT1W3","IMMT1W4","IMMT1W5",
    "IMMT2W1"
]

feature_cols = [f for f in qcp_features + moca_features if f in merged.columns]

print("\nSelected feature columns:", feature_cols)


# ---------------------------
# 6) Final Dataset
# ---------------------------
final_df = merged[["RID","VISCODE2","Diagnosis"] + feature_cols]
print("\nFinal dataset shape:", final_df.shape)

# Save for downstream ML / explainability
final_df.to_csv("Assessments/merged_final_dataset.csv", index=False)
print("\n✅ Final dataset saved -> Assessments/merged_final_dataset.csv")


Shapes -> EMBIC: (10398, 13)  MOCA: (8703, 58)  DXSUM: (15418, 41)
After merging -> (7036, 108)

Diagnosis value counts (raw):
Diagnosis
NaN    7036
Name: count, dtype: int64

Rows with valid Diagnosis: 0

Selected feature columns: ['RID', 'N1', 'N2', 'N3', 'N4', 'R1', 'R2', 'R3', 'M1', 'M2', 'M3', 'RHINO', 'REPEAT1', 'REPEAT2', 'MONTH', 'MOCA', 'TRAILS', 'CUBE', 'CLOCKCON', 'CLOCKNO', 'CLOCKHAN', 'LION', 'RHINO', 'CAMEL', 'IMMT1W1', 'IMMT1W2', 'IMMT1W3', 'IMMT1W4', 'IMMT1W5', 'IMMT2W1']

Final dataset shape: (0, 33)

✅ Final dataset saved -> Assessments/merged_final_dataset.csv


In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import shap
import matplotlib.pyplot as plt

# ---------------------------
# 1) Load datasets
# ---------------------------
embic = pd.read_csv("Assessments/EMBICqCP_08Jul2025.csv")
moca  = pd.read_csv("Assessments/MOCA_08Jul2025.csv")
dxsum = pd.read_csv("Assessments/DXSUM_08Jul2025.csv")

print("Shapes -> EMBIC:", embic.shape, " MOCA:", moca.shape, " DXSUM:", dxsum.shape)

# ---------------------------
# 2) Standardize column names
# ---------------------------
embic.columns = embic.columns.str.upper().str.strip()
moca.columns  = moca.columns.str.upper().str.strip()
dxsum.columns = dxsum.columns.str.upper().str.strip()

print("\n📌 Columns in DXSUM:\n", dxsum.columns.tolist()[:50], "...")  # print first 50 columns

# ---------------------------
# 3) Ensure keys exist
# ---------------------------
for df, name in [(embic,"EMBIC"), (moca,"MOCA"), (dxsum,"DXSUM")]:
    if not {"RID","VISCODE"}.issubset(df.columns):
        print(f"⚠️ {name} missing RID or VISCODE columns")

# ---------------------------
# 4) Merge all datasets
# ---------------------------
merged = pd.merge(embic, moca, on=["RID","VISCODE"], how="inner")
merged = pd.merge(merged, dxsum, on=["RID","VISCODE"], how="inner")

print("After merging ->", merged.shape)

# ---------------------------
# 5) Assign Diagnosis labels (CN / MCI / AD)
# ---------------------------
def assign_diagnosis(row):
    # Case 1: Binary flags exist
    if "DXNORM" in row and pd.notna(row["DXNORM"]) and row["DXNORM"] == 1:
        return "CN"
    if "DXMCI" in row and pd.notna(row["DXMCI"]) and row["DXMCI"] == 1:
        return "MCI"
    if ("DXAD" in row and pd.notna(row["DXAD"]) and row["DXAD"] == 1) or \
       ("DXOTHDEM" in row and pd.notna(row["DXOTHDEM"]) and row["DXOTHDEM"] == 1):
        return "AD"
    
    # Case 2: Single coded diagnosis column
    if "DXCHANGE" in row:
        code = row["DXCHANGE"]
        if code in [1, 7]:   # example: stable/normal
            return "CN"
        elif code in [2, 4, 8]:   # MCI codes
            return "MCI"
        elif code in [3, 5, 6, 9]:   # AD or other dementia
            return "AD"
    
    # Case 3: Column named DIAGNOSIS
    if "DIAGNOSIS" in row:
        val = str(row["DIAGNOSIS"]).upper()
        if "CN" in val:
            return "CN"
        elif "MCI" in val:
            return "MCI"
        elif "AD" in val or "DEMENTIA" in val:
            return "AD"
    
    return None

merged["Diagnosis"] = merged.apply(assign_diagnosis, axis=1)

print("Diagnosis counts after mapping:")
print(merged["Diagnosis"].value_counts(dropna=False))

# ---------------------------
# 6) Keep only valid Diagnosis rows
# ---------------------------
final = merged.dropna(subset=["Diagnosis"])
print("✅ Final dataset shape:", final.shape)
print("Diagnosis distribution:\n", final["Diagnosis"].value_counts())

if final.shape[0] == 0:
    raise ValueError("❌ No valid diagnosis rows found. Please re-check mapping rules.")

# ---------------------------
# 7) Select Features
# ---------------------------
feature_cols = [
    'N1','N2','N3','N4','R1','R2','R3',
    'M1','M2','M3','RHINO','REPEAT1','REPEAT2',
    'MONTH','MOCA','TRAILS','CUBE','CLOCKCON','CLOCKNO','CLOCKHAN',
    'LION','CAMEL','IMMT1W1','IMMT1W2','IMMT1W3','IMMT1W4','IMMT1W5',
    'IMMT2W1'
]

# keep only features that exist
feature_cols = [c for c in feature_cols if c in final.columns]
print("Selected feature columns:", feature_cols)

X = final[feature_cols].fillna(0)
y = final["Diagnosis"]

# ---------------------------
# 8) Train / Test Split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ---------------------------
# 9) Train RandomForest
# ---------------------------
clf = RandomForestClassifier(
    n_estimators=300, max_depth=None,
    random_state=42, class_weight="balanced"
)
clf.fit(X_train, y_train)

# ---------------------------
# 10) Evaluate
# ---------------------------
y_pred = clf.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ---------------------------
# 11) Explainable AI (SHAP)
# ---------------------------
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)

# Feature importance plot
plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("Feature Importance (SHAP)")
plt.show()

# Detailed SHAP summary
shap.summary_plot(shap_values, X_test)


Shapes -> EMBIC: (10868, 31)  MOCA: (8703, 58)  DXSUM: (15418, 41)

📌 Columns in DXSUM:
 ['PHASE', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE', 'DIAGNOSIS', 'DXNORM', 'DXNODEP', 'DXMCI', 'DXMDES', 'DXMPTR1', 'DXMPTR2', 'DXMPTR3', 'DXMPTR4', 'DXMPTR5', 'DXMPTR6', 'DXMDUE', 'DXMOTHET', 'DXDSEV', 'DXDDUE', 'DXAD', 'DXAPP', 'DXAPROB', 'DXAPOSS', 'DXPARK', 'DXPDES', 'DXPCOG', 'DXPATYP', 'DXDEP', 'DXOTHDEM', 'DXODES', 'DXCONFID', 'ID', 'SITEID', 'USERDATE', 'USERDATE2', 'DD_CRF_VERSION_LABEL', 'LANGUAGE_CODE', 'HAS_QC_ERROR', 'UPDATE_STAMP'] ...
After merging -> (7497, 126)
Diagnosis counts after mapping:
Diagnosis
None    7497
Name: count, dtype: int64
✅ Final dataset shape: (0, 127)
Diagnosis distribution:
 Series([], Name: count, dtype: int64)


ValueError: ❌ No valid diagnosis rows found. Please re-check mapping rules.

In [26]:
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# ---------------------------
# 1) Load datasets (use exact names from zip)
# ---------------------------
embic = pd.read_csv("Assessments/EMBIC_08Jul2025.csv")
moca  = pd.read_csv("Assessments/MOCA_08Jul2025.csv")
dxsum = pd.read_csv("Assessments/DXSUM_08Jul2025.csv")

print("EMBIC shape:", embic.shape)
print("MOCA shape:", moca.shape)
print("DXSUM shape:", dxsum.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'Assessments/EMBIC_08Jul2025.csv'

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import shap
import matplotlib.pyplot as plt

# ---------------------------
# 1) Load datasets
# ---------------------------
embic = pd.read_csv("Assessments/EMBICqCP_08Jul2025.csv")
moca  = pd.read_csv("Assessments/MOCA_08Jul2025.csv")
dxsum = pd.read_csv("Assessments/DXSUM_08Jul2025.csv")

print("Shapes -> EMBIC:", embic.shape, " MOCA:", moca.shape, " DXSUM:", dxsum.shape)

# ---------------------------
# 2) Standardize column names
# ---------------------------
embic.columns = embic.columns.str.upper().str.strip()
moca.columns  = moca.columns.str.upper().str.strip()
dxsum.columns = dxsum.columns.str.upper().str.strip()

print("\n📌 Columns in DXSUM:\n", dxsum.columns.tolist()[:50], "...")  # print first 50 columns

# ---------------------------
# 3) Ensure keys exist
# ---------------------------
for df, name in [(embic,"EMBIC"), (moca,"MOCA"), (dxsum,"DXSUM")]:
    if not {"RID","VISCODE"}.issubset(df.columns):
        print(f"⚠️ {name} missing RID or VISCODE columns")

# ---------------------------
# 4) Merge all datasets
# ---------------------------
merged = pd.merge(embic, moca, on=["RID","VISCODE"], how="inner")
merged = pd.merge(merged, dxsum, on=["RID","VISCODE"], how="inner")

print("After merging ->", merged.shape)

# ---------------------------
# 5) Assign Diagnosis labels (CN / MCI / AD)
# ---------------------------
def assign_diagnosis(row):
    # Case 1: Binary flags exist
    if "DXNORM" in row and pd.notna(row["DXNORM"]) and row["DXNORM"] == 1:
        return "CN"
    if "DXMCI" in row and pd.notna(row["DXMCI"]) and row["DXMCI"] == 1:
        return "MCI"
    if ("DXAD" in row and pd.notna(row["DXAD"]) and row["DXAD"] == 1) or \
       ("DXOTHDEM" in row and pd.notna(row["DXOTHDEM"]) and row["DXOTHDEM"] == 1):
        return "AD"
    
    # Case 2: Single coded diagnosis column
    if "DXCHANGE" in row:
        code = row["DXCHANGE"]
        if code in [1, 7]:   # example: stable/normal
            return "CN"
        elif code in [2, 4, 8]:   # MCI codes
            return "MCI"
        elif code in [3, 5, 6, 9]:   # AD or other dementia
            return "AD"
    
    # Case 3: Column named DIAGNOSIS
    if "DIAGNOSIS" in row:
        val = str(row["DIAGNOSIS"]).upper()
        if "CN" in val:
            return "CN"
        elif "MCI" in val:
            return "MCI"
        elif "AD" in val or "DEMENTIA" in val:
            return "AD"
    
    return None

merged["Diagnosis"] = merged.apply(assign_diagnosis, axis=1)

print("Diagnosis counts after mapping:")
print(merged["Diagnosis"].value_counts(dropna=False))

# ---------------------------
# 6) Keep only valid Diagnosis rows
# ---------------------------
final = merged.dropna(subset=["Diagnosis"])
print("✅ Final dataset shape:", final.shape)
print("Diagnosis distribution:\n", final["Diagnosis"].value_counts())

if final.shape[0] == 0:
    raise ValueError("❌ No valid diagnosis rows found. Please re-check mapping rules.")

# ---------------------------
# 7) Select Features
# ---------------------------
feature_cols = [
    'N1','N2','N3','N4','R1','R2','R3',
    'M1','M2','M3','RHINO','REPEAT1','REPEAT2',
    'MONTH','MOCA','TRAILS','CUBE','CLOCKCON','CLOCKNO','CLOCKHAN',
    'LION','CAMEL','IMMT1W1','IMMT1W2','IMMT1W3','IMMT1W4','IMMT1W5',
    'IMMT2W1'
]

# keep only features that exist
feature_cols = [c for c in feature_cols if c in final.columns]
print("Selected feature columns:", feature_cols)

X = final[feature_cols].fillna(0)
y = final["Diagnosis"]

# ---------------------------
# 8) Train / Test Split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ---------------------------
# 9) Train RandomForest
# ---------------------------
clf = RandomForestClassifier(
    n_estimators=300, max_depth=None,
    random_state=42, class_weight="balanced"
)
clf.fit(X_train, y_train)

# ---------------------------
# 10) Evaluate
# ---------------------------
y_pred = clf.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ---------------------------
# 11) Explainable AI (SHAP)
# ---------------------------
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)

# Feature importance plot
plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("Feature Importance (SHAP)")
plt.show()

# Detailed SHAP summary
shap.summary_plot(shap_values, X_test)


Shapes -> EMBIC: (10868, 31)  MOCA: (8703, 58)  DXSUM: (15418, 41)

📌 Columns in DXSUM:
 ['PHASE', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE', 'DIAGNOSIS', 'DXNORM', 'DXNODEP', 'DXMCI', 'DXMDES', 'DXMPTR1', 'DXMPTR2', 'DXMPTR3', 'DXMPTR4', 'DXMPTR5', 'DXMPTR6', 'DXMDUE', 'DXMOTHET', 'DXDSEV', 'DXDDUE', 'DXAD', 'DXAPP', 'DXAPROB', 'DXAPOSS', 'DXPARK', 'DXPDES', 'DXPCOG', 'DXPATYP', 'DXDEP', 'DXOTHDEM', 'DXODES', 'DXCONFID', 'ID', 'SITEID', 'USERDATE', 'USERDATE2', 'DD_CRF_VERSION_LABEL', 'LANGUAGE_CODE', 'HAS_QC_ERROR', 'UPDATE_STAMP'] ...
After merging -> (7497, 126)
Diagnosis counts after mapping:
Diagnosis
None    7497
Name: count, dtype: int64
✅ Final dataset shape: (0, 127)
Diagnosis distribution:
 Series([], Name: count, dtype: int64)


ValueError: ❌ No valid diagnosis rows found. Please re-check mapping rules.

In [2]:
# import pandas as pd
# import shap
# import matplotlib.pyplot as plt
# from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.preprocessing import LabelEncoder

# # -----------------------------
# # 1. Load dataset
# # -----------------------------
# df = pd.read_csv("alzheimers_scores.csv")   # Example file
# print(df.head())

# # Example columns: ['Age', 'Reaction_Time', 'Word_Recall_Score', 'Accuracy', 'Diagnosis']

# # Encode target labels
# le = LabelEncoder()
# df["Diagnosis"] = le.fit_transform(df["Diagnosis"])  # Healthy=0, MCI=1, AD=2

# X = df.drop("Diagnosis", axis=1)
# y = df["Diagnosis"]

# # -----------------------------
# # 2. Train ML model
# # -----------------------------
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model = RandomForestClassifier(n_estimators=200, random_state=42)
# model.fit(X_train, y_train)

# print("Model Accuracy:", model.score(X_test, y_test))

# # -----------------------------
# # 3. SHAP Explainability
# # -----------------------------
# explainer = shap.TreeExplainer(model)
# shap_values = explainer.shap_values(X_test)

# # -----------------------------
# # 4. Global Feature Importance
# # -----------------------------
# shap.summary_plot(shap_values, X_test, plot_type="bar")

# # -----------------------------
# # 5. Local Explanation (single patient)
# # -----------------------------
# # Pick one test sample (e.g., patient 5)
# sample_idx = 5
# shap.force_plot(
#     explainer.expected_value[1],  # baseline for class MCI
#     shap_values[1][sample_idx,:], 
#     X_test.iloc[sample_idx,:],
#     matplotlib=True
# )


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import shap
import matplotlib.pyplot as plt

# -----------------------------
# 1. Load and inspect dataset
# -----------------------------
df = pd.read_csv("Assessments/MemTrax.csv")
print(df.head())

# Expected columns:
# SubjectID, Trial, Stimulus (new/repeat), Response, Accuracy (0/1), ReactionTime

# -----------------------------
# 2. Feature engineering per subject
# -----------------------------
def extract_features(sub_df):
    features = {}
    features["Accuracy_Mean"] = sub_df["Accuracy"].mean()
    features["ReactionTime_Mean"] = sub_df["ReactionTime"].mean()
    features["ReactionTime_Std"] = sub_df["ReactionTime"].std()
    
    # Error types
    false_alarms = sub_df[(sub_df["Stimulus"]=="new") & (sub_df["Response"]=="seen")].shape[0]
    misses = sub_df[(sub_df["Stimulus"]=="repeat") & (sub_df["Response"]=="new")].shape[0]
    total_new = (sub_df["Stimulus"]=="new").sum()
    total_repeat = (sub_df["Stimulus"]=="repeat").sum()
    
    features["FalseAlarmRate"] = false_alarms / total_new if total_new > 0 else 0
    features["MissRate"] = misses / total_repeat if total_repeat > 0 else 0
    
    return pd.Series(features)

# Aggregate per subject
features_df = df.groupby("SubjectID").apply(extract_features).reset_index()

# -----------------------------
# 3. Labels (if available)
# -----------------------------
# Assume there’s a Diagnosis column mapping subjects to labels: Healthy / MCI / AD
labels = pd.read_csv("DiagnosisLabels.csv")  # e.g., SubjectID, Diagnosis
data = features_df.merge(labels, on="SubjectID")

# Encode target labels
le = LabelEncoder()
data["Diagnosis"] = le.fit_transform(data["Diagnosis"])

X = data.drop(["SubjectID", "Diagnosis"], axis=1)
y = data["Diagnosis"]

# -----------------------------
# 4. Train ML model
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

print("Model Accuracy:", model.score(X_test, y_test))

# -----------------------------
# 5. SHAP Explainability
# -----------------------------
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Global importance
shap.summary_plot(shap_values, X_test, plot_type="bar")

# Local explanation for first test subject
sample_idx = 0
shap.force_plot(
    explainer.expected_value[1], 
    shap_values[1][sample_idx,:], 
    X_test.iloc[sample_idx,:],
    matplotlib=True
)


FileNotFoundError: [Errno 2] No such file or directory: 'Assessments/MemTrax.csv'